# Токенизация

## Зачем нужна, какие виды, какие инструменты

<br>
Сашка Стукаленко 

## Вау, компьютер не умеет читать!

Вы знаете: модели машинного обучения работают с числами.

| Данные | Представление                                          |
|---|--------------------------------------------------------|
| Картинка | Тензор чисел (пиксели: яркость или цветовые каналы)    |
| Таблица | Числовая матрица (строки - объекты, столбцы -признаки) |
| **Текст** | **?**                                                  |

<br>

## Вау, компьютер не умеет читать!

Вы знаете: модели машинного обучения работают с числами.

| Данные    | Представление                                           |
|-----------|---------------------------------------------------------|
| Картинка  | Тензор чисел (пиксели: яркость или цветовые каналы)     |
| Таблица   | Числовая матрица (строки - объекты, столбцы - признаки) |
| **Текст** | Токенизация                                             |

<br>

**Токенизация** — процесс разбиения текста на минимальные единицы (токены), сопоставление числа каждому токену.

```
"кот сидит на крыше"  ->  [243, 891, 12, 564]
```

Мы переводим слова в **эмбеддинги** - векторы чисел, чтобы задать расстояние, и близкие по смыслу слова оказались рядом в числовом пространстве.
Именно с эмбеддингами уже и работает модель.

## Три уровня токенизации

Рассмотрим фразу `"нейросеть учится"`

| Уровень | Результат | Проблема |
|---|---|----------|
| **Word-level** | `["нейросеть", "учится"]` | ?        |
| **Character-level** | `["н","е","й","р","о","с","е","т","ь"," ",...]` | ?        |
| **Subword-level**  | `["ней", "##ро", "##сеть", "учится"]` | ?        |

<br>


## Три уровня токенизации

Рассмотрим фразу `"нейросеть учится"`

| Уровень | Результат | Проблема                                                         |
|---|---|------------------------------------------------------------------|
| **Word-level** | `["нейросеть", "учится"]` | Словарь огромен. Незнакомые слова - ошибка «нет в словаре» (OOV*). |
| **Character-level** | `["н","е","й","р","о","с","е","т","ь"," ",...]` | Последовательность слишком длинная (модель медленная).           |
| **Subword-level**  | `["ней", "##ро", "##сеть", "учится"]` | Компромисс: частые слова целиком, редкие разбиваются на части.   |

<br>

Subword кажется (и является) самым адекватным. Давайте рассмотрим на примере.

\* OOV = out-of-vocabulary — слово, которого нет в словаре модели.

## BPE: самый распространённый алгоритм

**Byte Pair Encoding** — лежит в основе GPT, LLaMA, RoBERTa и многих других моделях.

### Алгоритм BPE

1. Начинаем с символьного уровня
2. Находим самую частую пару соседних токенов
3. Объединяем -> новый токен
4. Повторяем до нужного размера словаря

Рассмотрим на примере фразы "низко низкий низость":
```
Шаг 1:  "н"+"и" -> "ни"   (встречается 3 раза)
Шаг 2:  "ни"+"з" -> "низ"
```
Повторяем процесс, пока не получим словарь нужного размера. В итоге алгоритм сам выучивает морфемы языка: корни, приставки, суффиксы.

## BPE: самый распространённый алгоритм
### В чём преимущества BPE?

1) Не требует лингвистических знаний - просто считает частоты.
2) Редкие слова разбиваются на частые кусочки -> нет неизвестных слов (никаких OOV).
3) Словарь компактный (тысячи токенов, а не миллионы), а последовательности не слишком длинные.


## GPT и Byte-level BPE токенизация любых символов

**Проблема классического BPE (на символах):**  
Он умеет разбивать редкие **слова** на частые подслова (нет OOV для слов). Но если в тексте встретится **символ**, которого не было в корпусе (то есть, изначальном наборе, на котором данные бились на токены), например, 撝, или ♥, то у токенизатора нет даже базового символа $\Rightarrow$ такой символ становится `[UNK]`.


## GPT и Byte‑level BPE: токенизация любых символов

**Проблема классического BPE (на символах):**  
Он умеет разбивать редкие **слова** на частые подслова (нет OOV для слов). Но если в тексте встретится **символ**, которого не было в корпусе (например, 擝, или ♥), то у токенизатора нет даже базового символа — такой символ становится `[UNK]`.

**Решение GPT:** применять BPE не к символам, а к **байтам** (UTF‑8).  
Байтов всего 256 — этого достаточно, чтобы закодировать **любой** символ. Алгоритм BPE работает с байтами так же, как с символами, но теперь **все возможные байты уже есть в словаре** (0–255). Следовательно, `[UNK]` исчезает полностью.

## GPT и Byte‑level BPE: что это даёт

| Что даёт байтовый BPE | Почему это важно |
|---|---|
| Словарь из ~50 000 токенов (байты + частые комбинации) | Компактно, умещается в память |
| Модель видит **любые** символы: эмодзи, код, арабский, иероглифы | **Абсолютно нет `[UNK]`** |
| Не нужно дообучать токенизатор под новый язык | Просто работает «из коробки» |


## GPT и Byte‑level BPE: токенизация любых символов

### Пример: токенизация фразы «Токенизируй меня 😊»


- Строка кодируется в байты UTF‑8.
- Байты группируются в частые пары по алгоритму BPE.
- Результат — токены, которые выглядят как кусочки слов и байтов:

```
«Токенизируй меня 😊»
→ токены:  ['Т', 'ок', 'ени', 'зи', 'руй', ' меня', '😊']
→ числа:   [789, 368, 337, 848, 28306, 703, 123]
```

Смайлик 😊 не разбивается — он может войти в словарь как целый токен (если часто встречается) или как последовательность байтов.


## GPT и Byte‑level BPE: токенизация любых символов

###  токен ≠ валидный символ!

Поскольку GPT использует byte-level токенизатор, некоторые токены в словаре не соответствуют ни одному осмысленному символу в UTF‑8. Они могут быть только частью многобайтовой последовательности. Например, в словаре GPT есть токены 167, 245 и 256. Попробуем их декодировать:
```
print(tokenizer.decode([167]))   # выводит: �
print(tokenizer.decode([245]))   # выводит: �
print(tokenizer.decode([256]))   # выводит: �
print(tokenizer.decode([167, 245, 256]))   # выводит: 撝
```
>Вывод: Модель не требует, чтобы каждый токен был осмысленным символом. Она учится комбинировать токены в более крупные единицы (символы, слова, идеи). Смысл рождается из последовательности, а не из отдельных элементов.

## WordPiece: алгоритм BERT

**WordPiece** — лежит в основе BERT, DistilBERT, Electra. Похож на BPE, но отличается критерием выбора пары.

### Алгоритм WordPiece

1. Начинаем с символьного уровня
2. Для каждой пары соседних токенов вычисляем **score**:
   `score = частота(пары) / (частота(A) × частота(B))`
3. Объединяем пару с **наибольшим** score
4. Повторяем до нужного размера словаря

Рассмотрим на примере «низко низкий низость»:
```
"н"+"и" встречается 3 раза. score = 3/(3×3) = 0.33
"з"+"к" встречается 2 раза. score = 2/(3×2) = 0.33
"н"+"и" -> "ни"    (выбираем по max score)
"ни"+"з" -> "низ"  (score вырос: "ни" редко встречается отдельно)
```

Алгоритм ищет **устойчивые связки** — пары, которые почти всегда идут вместе.

## Unigram LM: алгоритм SentencePiece

**Unigram Language Model** — лежит в основе SentencePiece (T5, XLNet, ALBERT). **Обратная логика** по сравнению с BPE.

### Алгоритм Unigram LM

1) Создаём максимальный словарь: все символы + все частые подстроки.

2) Обучаем вероятностную модель: для каждого токена считаем, как часто он встречается в корпусе, строим вероятности.

3) Оцениваем важность каждого токена: если мы удалим этот токен из словаря, то некоторые слова придётся разбивать на другие токены, и вероятность корпуса упадёт (loss вырастет). Считаем, на сколько вырастет loss.

4) Удаляем токены с наименьшим ростом loss

5) Повторяем шаги 2–4, пока словарь не уменьшится до нужного размера (например, 30 000 токенов).

Результат похож на BPE, но Unigram **оптимизирует вероятность** всего корпуса, а не просто частоту пар.

## Сравнение алгоритмов

| Алгоритм | Где используется | Принцип |
|---|---|---|
| BPE | GPT-2, RoBERTa | Слияние самых частых пар |
| Byte-level BPE | GPT-4, LLaMA, Mistral | BPE поверх байтов |
| WordPiece | BERT, DistilBERT | Слияние по правдоподобию |
| Unigram LM | T5, XLNet, ALBERT | Прореживание большого словаря |

## Когда токенизация ломает всё

**Пайплайн ML:** данные $\Rightarrow$ признаки $\Rightarrow$ модель $\Rightarrow$ результат.  
Для текста токенизация — это шаг «данные $\Rightarrow$ признаки».  
**GIGO (Garbage In, Garbage Out):** плохие признаки = плохая модель, даже если нейросеть огромная.

Токенизатор может уничтожить или исказить смысл ещё до того, как модель увидит данные.

### 1. Эмодзи: потеря важной части контекста

BERT (WordPiece, ~30k токенов) не видел большинства эмодзи → любой эмодзи заменяется на `[UNK]`.

| Исходная фраза | Что видит модель |
|---|---|
| `"You can break it 😞"` | `"You can break it [UNK]"` |
| `"You can't break it 😊"` | `"You can't break it [UNK]"` |

**Проблема:** противоположные смыслы стали **одинаковыми** для модели.  
**Решение:** байтовые токенизаторы (GPT, tiktoken, ~200k токенов) кодируют любые символы.

### 2. Даты: один смысл, разные токены

Модель не понимает, что все три записи — это одна и та же дата:

```
"20th October 2024"  →  ["20", "th", "October", "2024"]
"2024-10-20"         →  ["2024", "-", "10", "-", "20"]
"20/10/2024"         →  ["20", "/", "10", "/", "2024"]
```

Разные токены $\Rightarrow$ разные эмбеддинги $\Rightarrow$ модель не обобщает форматы.  
**Решение:** нормализовать даты до единого формата **до** токенизации — это задача предобработки.

### 3. Бренды и им подобные: потеря целостности

Редкий бренд разбивается на подслова из других контекстов:

```
"Gucci Savoy Leather-trimmed" → ["gu", "##cci", "Savoy", "Leather", "##trim", "##med"]
```

Поиск по `"Gucci"` не сработает: токена `"Gucci"` нет, есть только `"gu"` + `"##cci"`.  
**Решение:** добавить редкие сущности в словарь или использовать специальные токены.

---

## Вывод для практики

> **Токенизация — это не чёрный ящик.** Вы можете и должны контролировать, как текст превращается в числа.

Data cleaning для текста: нормализация дат, обработка эмодзи, защита брендов — это знакомый нам **EDA**, только для NLP.

Токенизатор — отдельный компонент, который нужно выбирать под вашу задачу.

### Сравнение популярных инструментов

| Инструмент | Алгоритм | Когда использовать | Почему именно он                                               |
|---|---|---|----------------------------------------------------------------|
| **NLTK** | Rule‑based (по пробелам/пунктуации) | Учёба, прототипы, курсовые | Простой, встроен в Python, не требует обучения                 |
| **SpaCy** | Rule‑based + ML (контекстные правила) | Продакшн, большие объёмы данных | Быстрый, промышленный, умеет выделять сущности                 |
| **HuggingFace Tokenizers** | BPE / WordPiece / Unigram | Работа с любым трансформером | Единый интерфейс, предобученные токенизаторы,очень быстро      |
| **tiktoken** | Byte‑level BPE (как в GPT‑4) | Работа с OpenAI API | Токенизатор как у моделей OpenAI  |
| **YouTokenToMe (VK)** | BPE, оптимизирован для многоязычия | Большие многоязычные корпуса | В 7–50 раз быстрее SentencePiece, меньше памяти                |


## Инструменты: главное правило

**Токенизатор и модель — единая система.**  
Нельзя взять токенизатор BERT и подать его результат в GPT, у них разные словари.  
Всегда используйте тот токенизатор, с которым обучалась конкретная модель.

```python
# AutoTokenizer сам подбирает нужный токенизатор под модель
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
```

## Токенизация универсальна

Мы говорили о тексте. Но принцип оказался настолько мощным, что сегодня он лежит в основе большинства направлений современного ИИ.

**Идея везде одна:** 
Взять данные любого типа → разбить на дискретные единицы → обработать трансформером.

### Изображения: Vision Transformer (ViT, Google Brain 2021)

Классические CNN обрабатывали изображения через свёртки. ViT делает иначе: изображение буквально нарезается на **патчи** 16×16 пикселей, и каждый патч становится токеном.

## ViT: как это работает


Изображение 224×224 пикселя  
    1) нарезаем на патчи 16×16 → 196 патчей  
    2) каждый патч разворачиваем в вектор (768 чисел)  
    3) добавляем позиционные эмбеддинги  
    4) 196 токенов подаём в стандартный трансформер


Статья "**An Image is Worth 16x16 Words**". Именно этот принцип открыл дорогу к мультимодальным моделям: GPT-4V, Gemini  работают с текстом и изображениями через единый механизм токенов.

## Токенизация вне NLP — продолжение

| Данные | Единица-токен | Пример | 
|-----|-------|------|
| Текст | Subword (BPE/WordPiece) | GPT, BERT |
| Изображение | Патч 16×16 пикселей | Vision Transformer (ViT) | 
| Аудио | Дискретный код ~25 мс | Whisper, EnCodec (Meta) | 
| Временной ряд | Окно из 64 точек | PatchTST | 
| RL-агент | (награда, состояние, действие) | Decision Transformer | 

<br>

Трансформер не меняется. Меняется только определение **единицы смысла** в конкретной области.

> "A Time Series is Worth 64 Words" статья PatchTST. Намеренная отсылка к ViT: авторы буквально показывают, что временной ряд и текст это одна и та же задача с точки зрения архитектуры.

## Будущее: токенизация как узкое место

Токенизация это полезный компромисс, но у неё есть ограничения.

**Проблема 1: Отдельное обучение без связи с задачей.**  
Токенизатор строится по статистике, т.е. он не знает, для какой задачи используется. Словарь, оптимальный для Wikipedia, может быть плохим для медицинских текстов.

**Проблема 2: Фиксированный словарь.**  
После обучения словарь фиксирован. Нельзя добавить новые слова, термины без переобучения.

**Проблема 3: Неравномерное качество по языкам.**  
BPE-словари обучены преимущественно на английском. То же предложение на русском требует в 2–3 раза больше токенов — дороже и хуже.  
Byte-level BPE смягчает это: любой байт уже есть в словаре, нет `[UNK]`. Но **слияния** всё равно натренированы на английских данных, русские буквы объединяются в меньшие, менее «выгодные» токены. Дисбаланс сохраняется.

## Будущее: ещё две проблемы

**Проблема 4: Каждый токен одинаково дорог.**  
Трансформер тратит одинаковое количество вычислений на токен «и» и на «квантовая», хотя частота применения невероятно отличается. Это неэффективно.

**Проблема 5: Модель не видит символов внутри токенов.**  
Это заслуживает отдельного слайда.

## «Strawberry Problem»

Сколько букв «r» в слове «strawberry»?
 
**GPT-4 отвечает: 2.**

Посмотрим, как токенизатор разбивает это слово:

```
"strawberry"  ->  ["st", "raw", "berry"]
```

Токен — неделимая единица для модели. Она не знает, какие буквы в него входят — только то, что он означает как целое. Это называют **дефицитом token awareness**: модель обрабатывает «berry» как единый блок, не имея доступа к его внутренней структуре.

Исследование *"Counting Ability of LLMs and Impact of Tokenization"* (2024) подтвердило это эмпирически: когда то же слово подавалось побуквенно через пробелы — точность подсчёта вырастала с 2–10% до 56–96%. Ошибки коррелируют с числом **токенов** в слове, а не с его длиной.

## «Strawberry Problem» — вывод

Тот же иероглиф 擝, который мы видели раньше, — другая сторона той же монеты. Смысл рождается из последовательности токенов. Но это же означает, что модель не может смотреть «внутрь» токена. Причина strawberry problem именно в токенизации, а не в недостатке «умственных способностей» модели.

## Альтернативы токенизации

Если токенизация — это компромисс, то каким может быть следующий шаг?

**ByT5 (Google Research, 2022) — работать с байтами напрямую**

Нет обучаемого словаря — модель работает напрямую с байтами (словарь фиксирован, 256 значений). Никаких OOV, никакого отдельного обучения — любой язык из коробки.  
Минус: последовательности длиннее — модель медленнее.

**Byte Latent Transformer (Meta AI, 2024) — умные патчи переменного размера**

Проблема ByT5: каждый байт стоит одинаково вычислений, даже если он часть тривиального «the». BLT решает это: байты группируются в патчи, размер которых зависит от **сложности** текста:

```
"the"         →  один патч    (3 байта → 1 вычисление, всё очевидно)
"Dosovitskiy" →  много патчей (12 байт → 12 вычислений, каждый важен)
```

Простой и частый текст обрабатывается быстро. Редкий и сложный — внимательнее. Ресурсы идут туда, где они нужны.

## Альтернативы: другое направление

**Large Concept Model (Meta AI, 2024) — работать со смыслом, а не с токенами**

ByT5 и BLT идут *вниз* — к байтам. LCM идёт в другую сторону: *вверх*, к целым предложениям.

Идея: каждое предложение кодируется в один вектор в «пространстве смыслов» (concept space). Модель не видит слов — она видит векторы-концепты:

```
"Идёт дождь."    →  вектор [0.31, -0.72, 0.44, ...]
"It is raining." →  вектор [0.31, -0.72, 0.44, ...]  ← почти тот же!
"Llueve."        →  вектор [0.31, -0.72, 0.44, ...]  ← и снова!
```

Один и тот же смысл на любом языке — один и тот же вектор. Мультиязычность без переобучения.

---

**Общий вектор:** токенизация — инженерный компромисс нынешнего поколения. Направление — либо убрать словарь (байтовый уровень), либо поднять абстракцию (концептуальный уровень).

## Итоговые тезисы



- Без токенизации нет NLP. Это первый и самый важный шаг от текста к числам.

- Subword-токенизация (BPE и др.) — стандарт. Не начинайте проект с word или character уровня, если цель не учебная.

- Проверяйте, как токенизатор обрабатывает эмодзи, даты, редкие бренды. Если они превращаются в [UNK] или разбиваются на нечитаемые куски — ищите другой токенизатор или делайте предобработку.

- Инструмент подбирайте под задачу: NLTK для быстрых экспериментов, SpaCy для скорости надёжности, HuggingFace для трансформеров, tiktoken для OpenAI.

- Принцип универсален: изображения делят на патчи, аудио на фреймы, везде один и тот же паттерн.

- Токенизация — инженерный компромисс нынешнего поколения. Что дальше? Либо убрать словарь (байтовый уровень), либо поднять абстракцию (концептуальный уровень).



## Источники — Хабр

1. [Краткий обзор токенизаторов](https://habr.com/ru/articles/800595/) — 2024
2. [Уделите внимание токенизаторам](https://habr.com/ru/articles/854664/) — 2024
3. [Почему токенизация — костыль?](https://habr.com/ru/articles/873120/) — 2025
4. [GPT для чайников](https://habr.com/ru/articles/599673/) — 2022
5. [YouTokenToMe от VK](https://habr.com/ru/companies/vk/articles/460641/) — 2019
6. [Основы NLP для текста](https://habr.com/ru/companies/Voximplant/articles/446738/) — 2019

## Источники — Научные статьи

7. [ViT: An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929) — Google Brain, ICLR 2021
8. [ByT5: Token-Free Future](https://arxiv.org/abs/2105.13626) — Google Research, 2022
9. [Decision Transformer](https://arxiv.org/abs/2106.01345) — 2021
10. [EnCodec: Neural Audio Compression](https://arxiv.org/abs/2210.13438) — Meta AI, 2022
11. [PatchTST: A Time Series is Worth 64 Words](https://arxiv.org/abs/2211.14730) — ICLR 2023
12. [Toward a Theory of Tokenization in LLMs](https://arxiv.org/abs/2404.08335) — UC Berkeley, 2024
13. [Counting Ability of LLMs and Tokenization](https://arxiv.org/abs/2410.19730) — 2024
14. [Why Do LLMs Struggle to Count Letters?](https://arxiv.org/abs/2412.18626) — 2024
15. [Byte Latent Transformer (BLT)](https://arxiv.org/abs/2412.09871) — Meta FAIR, 2024